# Experiment: freeze the truncation / representation constant

**Purpose.** The representations audit (deep_dive §2g) found models were fed inconsistent text and
the eligibility criteria were truncated off most rerankers. Before retraining anything, we pick ONE
representation strategy + `max_length` and justify it *in isolation*.

Two legs:
1. **Model-free coverage** — what fraction of the eligibility text each (strategy, max_length) actually
   preserves. No model, no training: the cleanest possible measure of the truncation constant itself.
2. **Reranker NDCG on a fixed pool** — feed the SAME judged pool to the current cross-encoder under
   each representation and measure TREC21 NDCG@10. (Caveat: the current clf was trained on `head`-512,
   so this leg has a train/inference-mismatch confound; the definitive reranker number comes after
   `retrain_classifier` on the frozen `R`. Use leg 1 to choose, leg 2 to sanity-check.)

Outcome: set `ExperimentConfig` defaults (`repr_strategy`, `max_length`, `head_frac`) that all
downstream notebooks inherit.


## Setup (Colab)


In [ ]:
# 1/3 — install the ctmatch package (includes ctmatch.experiments + the eval loader) + deps.
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets transformers accelerate pandas tqdm


In [ ]:
# 2/3 — mount Google Drive (holds the corpus, models, eval qrels).
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3/3 — env + import the backbone (ships inside the pip-installed ctmatch package).
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'          # big HF downloads stall on Colab's Xet backend
os.environ['HF_HOME'] = '/content/hf_cache'     # cache on fast local disk, not Drive

DATA_ROOT = '/content/drive/MyDrive/ct_data23'

import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from ctmatch.experiments import (ExperimentConfig, load_corpus, load_eval, eligibility_retained_frac, exclusion_retained_frac,
                                 cross_encoder_scores, relevant_index, ndcg_at_k, log_result)
print('backbone loaded; cuda:', torch.cuda.is_available())


In [ ]:
# Base config + the sweep grid. Only the representation varies; everything else is held fixed.
base = ExperimentConfig(data_root=DATA_ROOT)
STRATEGIES = ['head', 'head_tail', 'elig_first']   # 'head' = the current (eligibility-blind) deployed repr
LENGTHS    = [256, 384, 512]
HEAD_FRAC  = 0.5
GRID = [base.with_(repr_strategy=s, max_length=L, head_frac=HEAD_FRAC)
        for s in STRATEGIES for L in LENGTHS]
print('grid:', [c.repr_tag() for c in GRID])


In [ ]:
# Data: corpus fields (source of every representation) + TREC21 JUDGED POOL.
# The judged qrel docs per topic are a fixed candidate set that needs NO retrieval — ideal for
# isolating the reranker-INPUT representation (retrieval is held out of the picture entirely).
corpus_ids, corpus_fields = load_corpus(base)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(base, ['trec21'])
rel = sets['trec21']['rel_dict']; topic2text = sets['trec21']['topic2text']
pool = {t: list(rel[t]) for t in rel if t in topic2text}   # judged docs per topic
print('topics:', len(pool), '| mean judged pool size:', np.mean([len(v) for v in pool.values()]))


## Leg 1 — model-free eligibility coverage (evaluates the constant by itself)


In [ ]:
# For each grid point, what fraction of pool docs keep ALL / SOME of their eligibility text?
tok = AutoTokenizer.from_pretrained(base.clf_ckpt)
rows = []
for cfg in GRID:
    fracs = np.array([eligibility_retained_frac(tok, topic2text[t], id2fields[d], cfg)
                      for t, docs in pool.items() for d in docs if d in id2fields])
    rows.append({'repr': cfg.repr_tag(), 'mean_elig_retained': round(fracs.mean(), 3),
                 'pct_docs_full': round((fracs >= 0.999).mean(), 3),
                 'pct_docs_zero': round((fracs <= 0.001).mean(), 3)})
cov = pd.DataFrame(rows); cov


## Leg 1b — EXCLUSION-specific coverage
`mean_elig_retained` is exclusion-blind: `elig_first`/`head` keep inclusion *before* exclusion, so a long
inclusion list buries the disqualifier even when total eligibility coverage looks high. This measures the
exclusion span directly, and adds the `budget_incexc` arms (which reserve an exclusion floor).


In [ ]:
# Exclusion retention for the blob arms + budget_incexc arms (uses ctproc's inc/exc split per doc).
excl_arms = list(GRID) + [base.with_(repr_strategy='budget_incexc', max_length=L, budget_exc_frac=e)
                          for L in [384, 512] for e in [0.25, 0.40]]
xrows = []
for cfg in excl_arms:
    xr = np.array([exclusion_retained_frac(tok, topic2text[t], id2fields[d], cfg)
                   for t, docs in pool.items() for d in docs if d in id2fields])
    xrows.append({'repr': cfg.repr_tag(), 'mean_exc_retained': round(xr.mean(), 3),
                  'pct_docs_exc_full': round((xr >= 0.999).mean(), 3),
                  'pct_docs_exc_zero': round((xr <= 0.001).mean(), 3)})
pd.DataFrame(xrows)


**Read this table first.** If `head` (the deployed repr) shows a large `pct_docs_zero`, that is the
§2g defect quantified: the reranker never sees eligibility on those docs. `head_tail`/`elig_first`
should push `mean_elig_retained`→1.0. Coverage plateaus once eligibility fully fits — the smallest
`max_length` that keeps coverage ≈1.0 at acceptable cost is the constant to freeze.


## Leg 2 — reranker NDCG@10 on the judged pool (sanity check, with caveat)


In [ ]:
# Re-score the judged pool with the current clf under each representation; TREC21 NDCG@10.
# CAVEAT: clf was trained on head-512 -> mismatch for head_tail/elig_first. Definitive number
# comes after retrain_classifier on the frozen R. Here we just check the input effect isn't negative.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clf = AutoModelForSequenceClassification.from_pretrained(base.clf_ckpt).to(device).eval()
REL = relevant_index(clf)   # EXACT-match the 'relevant' class ('relevant' is a substring of 'not_relevant'!)

def rerank_ndcg(cfg):
    scores = []
    for t, docs in pool.items():
        docs = [d for d in docs if d in id2fields]
        s = cross_encoder_scores(clf, tok, topic2text[t], [id2fields[d] for d in docs], cfg, REL)
        ranked = [d for d, _ in sorted(zip(docs, s), key=lambda x: -x[1])]
        scores.append(ndcg_at_k(ranked, rel[t]))
    return float(np.mean(scores))

for cfg in GRID:
    nd = rerank_ndcg(cfg)
    log_result(cfg, experiment='truncation_study', split='trec21', metrics={'ndcg@10': nd})
    cov.loc[cov['repr'] == cfg.repr_tag(), 'rerank_ndcg@10'] = round(nd, 4)
cov


## Leg 2b — structured budget: topic / inclusion / exclusion (idea 1)
Reserves an exclusion floor (the disqualifier) instead of hoping truncation keeps it. inc/exc come
from ctproc's `process_eligibility_naive`. Compared to the head/head_tail/elig_first arms above.


In [ ]:
# budget_incexc arms: sweep the exclusion floor at the chosen length. Reuses rerank_ndcg from leg 2.
budget_arms = [base.with_(repr_strategy='budget_incexc', max_length=L,
                          budget_topic_frac=0.30, budget_exc_frac=e)
               for L in [384, 512] for e in [0.25, 0.40]]
brows = []
for cfg in budget_arms:
    nd = rerank_ndcg(cfg)
    log_result(cfg, experiment='truncation_study', split='trec21', metrics={'ndcg@10': nd})
    brows.append({'repr': cfg.repr_tag(), 'rerank_ndcg@10': round(nd, 4)})
pd.DataFrame(brows)


## Decision

Freeze the `ExperimentConfig` defaults to the winning `(repr_strategy, max_length, head_frac)`:
maximise eligibility coverage (leg 1) at the smallest length that doesn't cost reranker NDCG (leg 2),
trading off compute. Record the chosen `repr_tag` here; every downstream notebook inherits it via
`ExperimentConfig()` and every logged number is stamped with it. **Do not tune `max_length` again
after this** — that is the point of making it a constant.
